# Module 8: Explainable AI System

## NovaMart AI Retail Intelligence Platform

Objectives
- Explain Delivery Prediction model
- Explain Customer Satisfaction model
- Interpret feature importance using SHAP
- Explain individual predictions
- Generate visual explanations for the dashboard

### Overview

1. Introduction
2. Import Libraries
3. Load Models
4. Load Data
5. Delivery Prediction SHAP
6. Customer Satisfaction SHAP
7. Global Feature Importance
8. Local Explanation
9. Save SHAP Outputs
10. Business Insights

## Import Libraries

In [2]:
pip install shap

  Using cached shap-0.49.1-cp39-cp39-win_amd64.whl (547 kB)
  Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import joblib
import shap

plt.style.use("ggplot")

## Load Models

In [2]:
#Load Models
delivery_model = joblib.load(
    "random_forest_delivery_model.pkl"
)

delivery_features = joblib.load(
    "delivery_model_features.pkl"
)

satisfaction_model = joblib.load(
    "customer_satisfaction_model.pkl"
)

satisfaction_features = joblib.load(
    "customer_satisfaction_features.pkl"
)

In [3]:
import joblib

delivery_model = joblib.load("random_forest_delivery_model.pkl")

print("Delivery model loaded.")

Delivery model loaded.


In [4]:
satisfaction_model = joblib.load("customer_satisfaction_model.pkl")

print("Satisfaction model loaded.")

Satisfaction model loaded.


In [5]:
arima_model = joblib.load("arima_model.pkl")

print("ARIMA model loaded.")

ARIMA model loaded.


In [6]:
kmeans = joblib.load("models/customer_segmentation_kmeans.pkl")

print("KMeans loaded.")

KMeans loaded.


In [7]:
kmeans = joblib.load("customer_segmentation_kmeans.pkl")

print("KMeans loaded.")

KMeans loaded.


In [10]:
X_test = pd.read_csv("delivery_X_test.csv")
X_test.shape

(22366, 134)

## Load Dataset

In [ ]:
print(shap.__version__)

In [4]:
#Load Dataset
retail = pd.read_csv("retail_master_dataset.csv")

retail.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,review_comment_message,geolocation_zip_code_prefix,customer_lat,customer_lng,customer_geo_city,customer_geo_state,seller_lat,seller_lng,seller_geo_city,seller_geo_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,"Perfeito, produto entregue antes do combinado.",28013.0,-21.763186,-41.310265,campos dos goytacazes,RJ,-22.497188,-44.127324,volta redonda,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,NaN,15775.0,-20.222506,-50.898951,santa fe do sul,SP,-23.565754,-46.519097,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,Chegou antes do prazo previsto e o produto sur...,35661.0,-19.869998,-44.593059,para de minas,MG,-22.262802,-46.170735,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,NaN,12952.0,-23.105968,-46.590277,atibaia,SP,-20.553651,-47.387145,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,Gostei pois veio no prazo determinado .,13226.0,-23.243402,-46.827614,varzea paulista,SP,-22.929583,-53.135750,loanda,PR


## Prepare Delivery Dataset

In [5]:
delivery_df = retail.copy()

In [12]:
print(type(delivery_features))

<class 'list'>


In [16]:
print(delivery_features)

['price', 'freight_value', 'payment_value', 'payment_installments', 'processing_days', 'shipping_days', 'product_weight_g', 'product_volume_cm3', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'seller_state_BA', 'seller_state_CE', 'seller_state_DF', 'seller_state_ES', 'seller_state_GO', 'seller_state_MA', 'seller_state_MG', 'seller_state_MS', 'seller_state_MT', 'seller_state_PA', 'seller_state_PB', 'seller_state_PE', 'seller_state_PI', 'seller_state_PR', 'seller_state_RJ', 'seller_state_RN', 'seller_state_RO', 'seller_state_RS', 'seller_state_SC', 'seller_state_SE', 'seller_state_SP', 'customer_state_AL', 'customer_state_AM', 'customer_state_AP', 'customer_state_BA', 'customer_state_CE', 'customer_state_DF', 'customer_state_ES', 'customer_state_GO', 'customer_state_MA', 'customer_state_MG', 'customer_state_MS', 'customer_state_MT', 'customer_state_PA', 'customer_state_PB', 'customer_state_PE', 'customer_state_PI', 'customer_state_PR', 'customer_state_RJ', 'c

In [17]:
print(delivery_df.columns.tolist())

['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_value', 'payment_installments', 'payment_type', 'review_score', 'review_comment_title', 'review_comment_message', 'geolocation_zip_code_prefix', 'customer_lat', 'customer_lng', 'customer_geo_city', 'customer_geo_state', 'seller_lat', 'seller_lng', 'seller_geo_city', 'seller_geo_state']


In [18]:
missing = [col for col in delivery_features if col not in delivery_df.columns]

print("Missing columns:", missing)

Missing columns: ['processing_days', 'shipping_days', 'product_volume_cm3', 'seller_state_BA', 'seller_state_CE', 'seller_state_DF', 'seller_state_ES', 'seller_state_GO', 'seller_state_MA', 'seller_state_MG', 'seller_state_MS', 'seller_state_MT', 'seller_state_PA', 'seller_state_PB', 'seller_state_PE', 'seller_state_PI', 'seller_state_PR', 'seller_state_RJ', 'seller_state_RN', 'seller_state_RO', 'seller_state_RS', 'seller_state_SC', 'seller_state_SE', 'seller_state_SP', 'customer_state_AL', 'customer_state_AM', 'customer_state_AP', 'customer_state_BA', 'customer_state_CE', 'customer_state_DF', 'customer_state_ES', 'customer_state_GO', 'customer_state_MA', 'customer_state_MG', 'customer_state_MS', 'customer_state_MT', 'customer_state_PA', 'customer_state_PB', 'customer_state_PE', 'customer_state_PI', 'customer_state_PR', 'customer_state_RJ', 'customer_state_RN', 'customer_state_RO', 'customer_state_RR', 'customer_state_RS', 'customer_state_SC', 'customer_state_SE', 'customer_state_SP', 

In [6]:
X_delivery = delivery_df[delivery_features]

KeyError: "['processing_days', 'shipping_days', 'product_volume_cm3', 'seller_state_BA', 'seller_state_CE', 'seller_state_DF', 'seller_state_ES', 'seller_state_GO', 'seller_state_MA', 'seller_state_MG', 'seller_state_MS', 'seller_state_MT', 'seller_state_PA', 'seller_state_PB', 'seller_state_PE', 'seller_state_PI', 'seller_state_PR', 'seller_state_RJ', 'seller_state_RN', 'seller_state_RO', 'seller_state_RS', 'seller_state_SC', 'seller_state_SE', 'seller_state_SP', 'customer_state_AL', 'customer_state_AM', 'customer_state_AP', 'customer_state_BA', 'customer_state_CE', 'customer_state_DF', 'customer_state_ES', 'customer_state_GO', 'customer_state_MA', 'customer_state_MG', 'customer_state_MS', 'customer_state_MT', 'customer_state_PA', 'customer_state_PB', 'customer_state_PE', 'customer_state_PI', 'customer_state_PR', 'customer_state_RJ', 'customer_state_RN', 'customer_state_RO', 'customer_state_RR', 'customer_state_RS', 'customer_state_SC', 'customer_state_SE', 'customer_state_SP', 'custo

## SHAP for Delivery Prediction

Create the explainer.

In [ ]:
delivery_explainer = shap.TreeExplainer(
    delivery_model
)

delivery_shap = delivery_explainer.shap_values(
    X_delivery
)

## Global Feature Importance

In [ ]:
shap.summary_plot(

    delivery_shap[1],

    X_delivery

)

## Waterfall Plot (Single Prediction)

In [ ]:
sample = 0

shap.plots.waterfall(

    shap.Explanation(

        values=delivery_shap[1][sample],

        base_values=delivery_explainer.expected_value[1],

        data=X_delivery.iloc[sample],

        feature_names=X_delivery.columns

    )

)

## Force Plot

In [ ]:
shap.initjs()

shap.force_plot(

    delivery_explainer.expected_value[1],

    delivery_shap[1][sample],

    X_delivery.iloc[sample]

)

## Customer Satisfaction

In [ ]:
satisfaction_df = retail.copy()


In [ ]:
# Repeat preprocessing from Notebook 06.

X_satisfaction = satisfaction_df[
    satisfaction_features
]

## Create SHAP values.

In [ ]:
satisfaction_explainer = shap.TreeExplainer(
    satisfaction_model
)

satisfaction_shap = satisfaction_explainer.shap_values(
    X_satisfaction
)

In [ ]:
shap.summary_plot(

    satisfaction_shap[1],

    X_satisfaction

)

In [ ]:
shap.summary_plot(

    satisfaction_shap[1],

    X_satisfaction,

    plot_type="bar"

)

In [ ]:
sample = 3

shap.plots.waterfall(

    shap.Explanation(

        values=satisfaction_shap[1][sample],

        base_values=satisfaction_explainer.expected_value[1],

        data=X_satisfaction.iloc[sample],

        feature_names=X_satisfaction.columns

    )

)

## Dependence Plot

Example:

In [ ]:
shap.dependence_plot(

    "delivery_days",

    delivery_shap[1],

    X_delivery

)

In [ ]:
shap.dependence_plot(

    "payment_value",

    satisfaction_shap[1],

    X_satisfaction

)

## Save SHAP Objects

In [ ]:
joblib.dump(

    delivery_explainer,

    "../models/delivery_shap_explainer.pkl"

)

joblib.dump(

    satisfaction_explainer,

    "../models/satisfaction_shap_explainer.pkl"

)

print("SHAP explainers saved successfully.")

## Save SHAP Values

In [ ]:
Save SHAP Values
np.save(

    "../data/delivery_shap_values.npy",

    delivery_shap[1]

)

np.save(

    "../data/satisfaction_shap_values.npy",

    satisfaction_shap[1]

)

print("SHAP values saved.")